# Notebook 27 — Where does the signal work?

**Purpose:** Diagnose whether pooled price/volume classifiers conceal ticker, sector, or regime-specific edges, and whether a cost-aware target is more useful. **Development-only:** the historical holdout and prospective ledger remain untouched.

**Important:** outer-fold segment/decile comparisons are exploratory; do not promote a subgroup or confidence threshold selected from these same folds. The economic binary target is a separate prediction task and its accuracy is **not directly comparable** with the Q7 three-class accuracy.


In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if not (ROOT/'src').exists(): raise RuntimeError('Open notebook from repository root or notebooks directory')
if str(ROOT/'src') not in sys.path: sys.path.insert(0,str(ROOT/'src'))
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.decision_diagnostics import run_decision_diagnostics
import pandas as pd
import numpy as np
print('Project:',ROOT)


## 1. Research configuration

Six pooled configurations per fold (Random Forest, Logistic Regression, and ticker-aware Logistic Regression; each with price-only and price+volume). Additional sector-specific Logistic Regression experiments run **only when real sector metadata exists**. Three chronological folds, 10 confidence deciles per configuration, ticker and regime diagnostics, confusion matrices, and a separately labeled economic binary target.


In [ ]:
N_FOLDS=3
MAX_TRAIN_ROWS=50000
TREE_TRAIN_ROWS=35000
COST_BPS=10
INCLUDE_VOLUME=True
OUTPUT=ROOT/'artifacts'/'decision_diagnostics'/'1d'
print('Expected pooled model fits:',N_FOLDS*6)


## 2. Load frozen master data

Always unpack all four values returned by `load_master()`.


In [ ]:
required=('master_labeled.parquet','latest_inference.parquet','transformers.joblib','manifest.json')
paths=[ROOT/'artifacts'/'master_tables'/'1d',ROOT/'notebooks'/'artifacts'/'master_tables'/'1d']
source=next((p for p in paths if all((p/f).exists() for f in required)),None)
if source is None: raise FileNotFoundError('Master artifacts missing; run Notebook 03')
master,latest,bundle,manifest=load_master(source)
print('Rows:',len(master),'symbols:',master.symbol.nunique(),'dates:',master.bar_timestamp.min(),master.bar_timestamp.max())
print('Available sector columns:',[c for c in ('sector','sector_name','gics_sector','sector_code','sector_id') if c in master])


## 3. Run research

Expect a longer execution time. Features, target quantiles, and preprocessing are fitted/constructed causally. The volatility-regime cutoff is estimated using each fold's training period only. Sector experiments are skipped if no trustworthy sector column is present.


In [ ]:
outputs,cutoff,sector=run_decision_diagnostics(master,n_folds=N_FOLDS,
    max_train_rows=MAX_TRAIN_ROWS,tree_train_rows=TREE_TRAIN_ROWS,
    cost_bps=COST_BPS,include_volume=INCLUDE_VOLUME,output_dir=OUTPUT)
print('Development cutoff:',cutoff,'sector metadata:',sector or 'not available')
print('Saved CSV diagnostics to:',OUTPUT)


## 4. Model performance: paired chronological folds

Use the same model and fold to compare price-only against price+volume. Examine fold-by-fold lift, not just averages.


In [ ]:
scores=outputs['scores']
display(scores.sort_values(['model','fold']).round(4))
pooled=scores[scores.segment.eq('all')]
display(pooled.groupby('model').agg(accuracy=('accuracy','mean'),baseline=('majority_baseline','mean'),
    auc=('auc','mean'),entry_precision=('entry_precision','mean'),entry_coverage=('entry_coverage','mean'),
    entry_net=('entry_mean_net','mean'),folds=('fold','nunique')).sort_values('accuracy',ascending=False).round(4))


## 5. Sector and ticker heterogeneity

Ticker rows are descriptive and have a minimum of 30 observations per fold. **Do not rank tickers for live trading from these results alone.** Sector-specific training requires an existing sector field; no mapping is fabricated.


In [ ]:
display(outputs['ticker_diagnostics'].sort_values(['fold','entry_mean_net'],ascending=[True,False]).head(35).round(4))
if sector:
    display(scores[scores.model.eq('sector_logistic')].sort_values(['fold','segment_value']).round(4))
else:
    print('Sector-specific model skipped: master table has no sector metadata. Join point-in-time GICS reference in a future controlled experiment.')


## 6. Market regimes: does the edge persist?

High/low volatility uses a training-fold median. Breadth and momentum use current-bar signs. These are **diagnostic slices**, not strategies.


In [ ]:
segments=outputs['segments']
for segment in ('volatility_regime','breadth_regime','momentum_regime'):
    print('\n',segment)
    display(segments[segments.segment.eq(segment)].groupby(['model','segment_value']).agg(
        n=('n','sum'),accuracy=('accuracy','mean'),entry_precision=('entry_precision','mean'),
        entry_coverage=('entry_coverage','mean'),entry_net=('entry_mean_net','mean')).round(4))


## 7. Probability calibration and confidence deciles

These deciles are ranked within each test fold. Look for monotonic increases in observed Up prevalence and net return, with meaningful sample counts. Do not interpret an extreme decile with few entries as validated.


In [ ]:
dec=outputs['deciles']
display(dec.groupby(['model','decile']).agg(n=('n','sum'),p_up=('mean_p_up','mean'),
    observed_up=('observed_up','mean'),next_return=('mean_next_return','mean'),
    net_long=('mean_net_long','mean'),brier=('up_brier','mean')).round(4))


## 8. Economic target: a distinct prediction task

Binary outcome: next return exceeds the assumed **10 bps** cost. Report its own majority baseline and ROC-AUC. Accuracy across this binary task and Q7 cannot be directly compared. The 10 bps cost is a simplifying assumption, not a calibrated estimate of spread, slippage, and market impact.


In [ ]:
display(outputs['economic_targets'].round(4))


## 9. Confusion matrices and compute budget


In [ ]:
conf=outputs['confusions']
for name in sorted(conf.model.unique()):
    print(name)
    display(conf[conf.model.eq(name)].pivot_table(index='true_class',columns='pred_class',values='n',aggfunc='sum',fill_value=0))
display(outputs['timings'].groupby('model').agg(total_fit_seconds=('train_seconds','sum'),mean_train_rows=('train_rows','mean')).round(2))


## 10. Decision criteria for Notebook 28

- **Sector specialization:** pursue only if a real sector mapping exists and gains recur across folds with adequate sector sample sizes.
- **Ticker personalization:** require paired improvement over pooled price models and stability; ticker-specific leaderboard winners are exploratory.
- **Volume:** retain only when improvements recur in at least two of three folds and have nontrivial magnitude.
- **Economic target:** compare against its **own** majority baseline and examine calibration, coverage, and returns.
- **Entry policy:** no promotion from this notebook. A policy must be frozen before an independent prospective shadow evaluation.

**Do not reopen the historical holdout or modify the prospective prediction ledger.**
